In [ ]:
import torch
import os
import time
import webdataset as wds
from pathlib import Path
import torch.distributed as dist
from siglip import loss_fn, get_lr
from utils import filter_sample, preprocess, save_checkpoint, model_import
from DDPManager import DDPManager
from functools import partial
from val import eval_model

device = (
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

torch.manual_seed(1337)
if device == "cuda":
    torch.cuda.manual_seed(1337)

In [2]:
url = "https://huggingface.co/datasets/pixparse/cc12m-wds/resolve/main/cc12m-train-{0000..0003}.tar"
val_url = "https://huggingface.co/datasets/pixparse/cc12m-wds/resolve/main/cc12m-train-{0004..0007}.tar"
train_url = "https://huggingface.co/datasets/pixparse/cc12m-wds/resolve/main/cc12m-train-{0000..2100}.tar"
eval_url = "https://huggingface.co/datasets/pixparse/cc12m-wds/resolve/main/cc12m-train-{2101..2150}.tar"
test_url = "https://huggingface.co/datasets/pixparse/cc12m-wds/resolve/main/cc12m-train-{2151..2175}.tar"

In [3]:
train_cache_dir = Path("train_cache_dir")
train_cache_dir.mkdir(parents=True, exist_ok=True)
trainset = (
    wds.WebDataset(
        url,
        shardshuffle=False,
        resampled=True,
        cache_dir=train_cache_dir,
        nodesplitter=wds.split_by_node,
    )  # type: ignore
    .shuffle(1000)
    .decode("pil")
    .select(filter_sample)
    .batched(512)
    .map(preprocess)
)

sample = next(iter(trainset))
for key, value in sample.items():
    if key != "__key__":
        print(f"{key}: {value.shape}")

trainloader = wds.WebLoader(
    trainset, batch_size=None, num_workers=1, pin_memory=True, prefetch_factor=2
)  # type: ignore
trainloader = trainloader.unbatched().shuffle(100).batched(32)


for sample in trainloader:
    print(sample["attention_mask"].shape)
    print(sample["input_ids"].shape)
    print(sample["pixel_values"].shape)
    break

pixel_values: torch.Size([512, 3, 256, 256])
input_ids: torch.Size([512, 64])
attention_mask: torch.Size([512, 64])


/Users/mr.fearless/ABHAY/AI/OpenWorld/Siglip/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


torch.Size([32, 64])
torch.Size([32, 64])
torch.Size([32, 3, 256, 256])


In [4]:
val_cache_dir = Path("val_cache_dir")
val_cache_dir.mkdir(parents=True, exist_ok=True)
evalset = (
    wds.WebDataset(
        val_url,
        shardshuffle=False,
        cache_dir=val_cache_dir,
        nodesplitter=wds.split_by_node,
    )  # type: ignore
    .decode("pil")
    .select(filter_sample)
    .batched(64)
    .map(preprocess)
)
evalloader = wds.WebLoader(
    evalset, batch_size=None, num_workers=4, pin_memory=True, prefetch_factor=4
)  # type: ignore

for sample in evalloader:
    print(sample["attention_mask"].shape)
    print(sample["input_ids"].shape)
    print(sample["pixel_values"].shape)
    break

torch.Size([64, 64])
torch.Size([64, 64])
torch.Size([64, 3, 256, 256])


In [5]:
model = model_import(device)
# torch.cuda.empty_cache()

Starting transfer of weights from hf_model to my_model
for text embedding layers <All keys matched successfully>
for layer 0 <All keys matched successfully>
for layer 1 <All keys matched successfully>
for layer 2 <All keys matched successfully>
for layer 3 <All keys matched successfully>
for layer 4 <All keys matched successfully>
for layer 5 <All keys matched successfully>
for layer 6 <All keys matched successfully>
for layer 7 <All keys matched successfully>
for layer 8 <All keys matched successfully>
for layer 9 <All keys matched successfully>
for layer 10 <All keys matched successfully>
for layer 11 <All keys matched successfully>
for final norm layer <All keys matched successfully>
for final head <All keys matched successfully>


In [6]:
ddp = DDPManager()
device = ddp.device
model = ddp.wrap_model(model)
model = torch.compile(model)
# ddp.cleanup()
raw_model = model.module if ddp.is_distributed else model  # type: ignore

num_samples = 10240000  # 10586100
global_batch_size = 256
mini_batch_size = 32
num_epochs = 2
num_shards = 2100
num_sample = 5000
world_size = 1  # ddp.world_size  # 2-4

grad_accum_steps = int(global_batch_size / (mini_batch_size * world_size))
max_steps = int(num_epochs * (num_samples / global_batch_size) * grad_accum_steps)
warmup_steps = int(0.2 * max_steps)

max_lr = 1e-3
min_lr = max_lr * 0.1  # might be 0.01

trainloader = trainloader.with_epoch(max_steps)
val_loader = evalloader.with_epoch(8)
optimizer = model.configure_optimizers(weight_decay=0.05, learning_rate=max_lr)  # type: ignore

Initialized: Distributed=False, World Size=1, Rank=0, Device=mps
710.7714648979029
num decayed parameter tensors: 55, with 92,799,744 parameters
num non-decayed parameter tensors: 107, with 130,562 parameters


In [8]:
prefilled_lr = partial(get_lr, warmup_steps, max_steps, max_lr, min_lr)
optimizer.zero_grad(set_to_none=True)
opt_step = 0
t0 = time.time()


try:
    for step, batch in enumerate(trainloader):
        if opt_step >= 10:
            break
        update_step = (step + 1) % grad_accum_steps == 0

        if ddp.is_distributed:
            model.require_backward_grad_sync = update_step  # type: ignore

        pixel_values = batch["pixel_values"].to(device, non_blocking=True)
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)

        logits = model(input_ids, attention_mask, pixel_values)
        loss = loss_fn(logits) / grad_accum_steps
        loss.backward()

        if not update_step:
            continue

        loss_to_log = loss.detach() * grad_accum_steps
        if ddp.is_distributed:
            dist.all_reduce(loss_to_log, op=dist.ReduceOp.AVG)

        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # type: ignore
        lr = prefilled_lr(opt_step)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        opt_step += 1

        t1 = time.time()
        dt = t1 - t0
        t0 = t1

        if ddp.is_master:
            logit_scale = raw_model.logit_scale.exp()  # type: ignore
            pairs = mini_batch_size * grad_accum_steps * world_size
            pairs_per_sec = pairs / dt
            log_str = (
                f"opt_step={opt_step} | step={step + 1} | "
                f"loss={loss_to_log.item():.6f} | lr={lr:.6f} | "
                f"norm={norm.item():.4f} | dt={dt:.4f}s | "
                f"pairs/sec={pairs_per_sec:.2f} | logit_scale={logit_scale.item():.4f}"
            )
            print(log_str)
            with open("logs/training.log", "a") as f:
                f.write(f"{log_str}\n")

        if opt_step % 5 == 0:
            if opt_step >= 5:
                save_checkpoint(raw_model, optimizer, opt_step, lr)
            model.eval()  # type: ignore
            val_loss, val_r1, val_r5, val_margin = eval_model(
                model, val_loader, device, loss_fn, ddp.is_distributed
            )
            ddp.barrier()
            if ddp.is_master:
                val_str = (
                    f"[VAL] loss={val_loss:.6f} | R@1={val_r1:.4f} | "
                    f"R@5={val_r5:.4f} | margin={val_margin:.4f} | opt_step={opt_step}"
                )
                print(val_str)
                with open("logs/validation.log", "a") as f:
                    f.write(f"{val_str}\n")
            model.train()  # type: ignore

except Exception as e:
    if ddp.is_master:
        print(f"Error at step {step}: {e}")
        save_checkpoint(raw_model, optimizer, opt_step, lr)
    raise

step=8 | opt_step=1 | loss=9.948628 | lr=0.000000 | norm=48.8286 | dt=48.5375s | pairs/sec=5.27 | logit_scale=10.0000
step=16 | opt_step=2 | loss=9.934743 | lr=0.000000 | norm=48.3015 | dt=39.5482s | pairs/sec=6.47 | logit_scale=10.0000
step=24 | opt_step=3 | loss=9.972895 | lr=0.000000 | norm=48.7788 | dt=40.0508s | pairs/sec=6.39 | logit_scale=10.0000
step=32 | opt_step=4 | loss=9.959562 | lr=0.000000 | norm=48.0139 | dt=43.4180s | pairs/sec=5.90 | logit_scale=10.0000
step=40 | opt_step=5 | loss=9.919701 | lr=0.000000 | norm=50.7560 | dt=47.9255s | pairs/sec=5.34 | logit_scale=10.0000
✅ Checkpoint saved at step 5 to checkpoints/siglip_step_5.pt


/Users/mr.fearless/ABHAY/AI/OpenWorld/Siglip/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
W0120 11:38:30.074000 80381 torch/_inductor/utils.py:1613] [0/1] Not enough SMs to use max_autotune_gemm mode


[VAL] loss=9.965461 | R@1=0.0020 | R@5=0.0078 | margin=-0.0053 | opt_step=5
step=48 | opt_step=6 | loss=9.877647 | lr=0.000000 | norm=49.6748 | dt=145.7793s | pairs/sec=1.76 | logit_scale=10.0000
step=56 | opt_step=7 | loss=10.055491 | lr=0.000000 | norm=48.2847 | dt=47.1484s | pairs/sec=5.43 | logit_scale=10.0000
step=64 | opt_step=8 | loss=10.018148 | lr=0.000000 | norm=50.1378 | dt=51.6703s | pairs/sec=4.95 | logit_scale=10.0000
step=72 | opt_step=9 | loss=9.957033 | lr=0.000000 | norm=48.1677 | dt=53.3954s | pairs/sec=4.79 | logit_scale=10.0000
step=80 | opt_step=10 | loss=9.844021 | lr=0.000000 | norm=50.8206 | dt=51.9738s | pairs/sec=4.93 | logit_scale=10.0000
✅ Checkpoint saved at step 10 to checkpoints/siglip_step_10.pt


/Users/mr.fearless/ABHAY/AI/OpenWorld/Siglip/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


[VAL] loss=9.935540 | R@1=0.0020 | R@5=0.0059 | margin=-0.0053 | opt_step=10
